In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dense, Flatten, BatchNormalization, Conv2D, MaxPool2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix
import itertools
import os
import shutil
import random
import glob
import matplotlib.pyplot as plt
import warnings
import pandas as pd
warnings.simplefilter(action='ignore', category=FutureWarning)
%matplotlib inline

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("Using CPU. Enable a GPU runtime for acceleration if available.")

In [ ]:
IMG_SIZE=128 # keep the size same as the saved trained model was trained upon the image size

In [ ]:
model = Sequential([
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding = 'same', input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPool2D(pool_size=(2, 2), strides=2),
    Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding = 'same'),
    MaxPool2D(pool_size=(2, 2), strides=2),
    Flatten(),
    Dense(units=43, activation='softmax')
])

### Load saved model - 

1. From drive/MyDrive/traffic_sign_model.keras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model = tf.keras.models.load_model(
    '/content/drive/MyDrive/traffic_sign_model.keras'
)

print("Trained model loaded successfully!")

2. Load it from /Models/gtsrb_cnn.keras

In [ ]:
model = tf.keras.models.load_model(
    '/Models/gtsrb_cnn.keras'
)

print("Trained model loaded successfully!")

# Test a custom image

In [ ]:
from pathlib import Path
from tensorflow.keras.utils import load_img, img_to_array
from IPython.display import display

image_dir = Path("/content/assets")

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_files = sorted(
    p for p in image_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
)

if not image_files:
    raise FileNotFoundError(
        f"No images found in {image_dir}. Place your test image there."
    )

image_path = image_files[0]

print("Testing image:", image_path)

display(load_img(str(image_path)))

img = load_img(
    str(image_path),
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="rgb"
)

img_array = img_to_array(img)
img_array = tf.keras.applications.vgg16.preprocess_input(img_array)
img_array = np.expand_dims(img_array, axis=0)

predictions = model.predict(img_array, verbose=0)[0]

predicted_class = np.argmax(predictions)
confidence = predictions[predicted_class] * 100

print("\nPredicted ClassId:", predicted_class)
print(f"Confidence: {confidence:.2f}%")

print("\nTop 5 predictions:")

for class_id in np.argsort(predictions)[-5:][::-1]:
    print(f"ClassId {class_id}: {predictions[class_id] * 100:.2f}%")